In [19]:
import sys
import os
import pandas as pd
from pathlib import Path

In [20]:
# Project setup
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data" / "raw").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

- Show what columns exist and what a few rows look like
- Followed this as a rule of thumb: peek with pandas, process with Spark.

In [54]:
for name in ["air_quality", "weather", "taxi_zones"]:
    folder = Path("data/raw") / name
    files = sorted(folder.glob("*.csv"))
    print(f"--{name}: {[f.name for f in files]}--")

    df = pd.read_csv(files[0])
    print("Columns:", list(df.columns))
    print(df.head(3))
    print()

--air_quality: ['hourly_88101_2024.csv']--


C:\Users\user\AppData\Local\Temp\ipykernel_15012\2954518178.py:6: DtypeWarning: Columns (0: Qualifier) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(files[0])


Columns: ['State Code', 'County Code', 'Site Num', 'Parameter Code', 'POC', 'Latitude', 'Longitude', 'Datum', 'Parameter Name', 'Date Local', 'Time Local', 'Date GMT', 'Time GMT', 'Sample Measurement', 'Units of Measure', 'MDL', 'Uncertainty', 'Qualifier', 'Method Type', 'Method Code', 'Method Name', 'State Name', 'County Name', 'Date of Last Change']
   State Code  County Code  Site Num  Parameter Code  POC   Latitude  \
0           1            3        10           88101    3  30.497478   
1           1            3        10           88101    3  30.497478   
2           1            3        10           88101    3  30.497478   

   Longitude  Datum            Parameter Name  Date Local  ...  \
0 -87.880258  NAD83  PM2.5 - Local Conditions  2024-01-02  ...   
1 -87.880258  NAD83  PM2.5 - Local Conditions  2024-01-02  ...   
2 -87.880258  NAD83  PM2.5 - Local Conditions  2024-01-02  ...   

              Units of Measure  MDL Uncertainty  Qualifier Method Type  \
0  Micrograms/cubi

In [56]:
from src.common.spark_session import get_spark

spark = get_spark()

trips = spark.read.parquet("data/raw/taxi_trips")
print("Rows:", trips.count())
trips.printSchema()
trips.show(3, truncate=False)

Rows: 9554778
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+-----------------

### Checking for candiate Primary Keys

In [58]:
from pyspark.sql import functions as F

taxi_key = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID"
]

trips.groupBy(taxi_key).count() \
    .filter(F.col("count") > 1) \
    .show(10)

+--------+--------------------+---------------------+------------+------------+-----+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|count|
+--------+--------------------+---------------------+------------+------------+-----+
|       2| 2024-01-01 00:04:00|  2024-01-01 00:04:44|          63|          63|    2|
|       2| 2024-01-01 00:42:02|  2024-01-01 01:14:33|         107|          61|    2|
|       2| 2024-01-01 00:38:59|  2024-01-01 00:40:05|          68|          68|    2|
|       2| 2024-01-01 00:35:48|  2024-01-01 00:47:28|         234|         125|    2|
|       2| 2024-01-01 00:40:25|  2024-01-01 00:46:13|          68|          68|    2|
|       2| 2024-01-01 00:46:42|  2024-01-01 01:34:57|           4|         143|    2|
|       2| 2024-01-01 00:28:40|  2024-01-01 00:50:44|         229|         129|    2|
|       2| 2024-01-01 00:18:50|  2024-01-01 00:35:33|         148|         170|    2|
|       2| 2024-01-01 01:14:09|  2024-01-01 02:00:27| 

In [64]:
trips.groupBy(taxi_key).count() \
    .filter(F.col("count") > 1) \
    .count()

101646

In [65]:
trips.filter(
    (F.col("VendorID") == 2) &
    (F.col("tpep_pickup_datetime") == "2024-01-01 00:04:00") &
    (F.col("tpep_dropoff_datetime") == "2024-01-01 00:04:44") &
    (F.col("PULocationID") == 63) &
    (F.col("DOLocationID") == 63)
).show(truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:04:00 |2024-01-01 00:04:44  |1              |0.01         |5         |N                 |63          |63          |2           |-31.5      |0.0  |0.0    |0.0      

- Primary key: No explicit primary key is provided in the dataset. The available attributes do not guarantee unique identification of individual trips.
- The Taxi Trips dataset contains multiple records sharing the same apparent trip-identifying attributes, so uniqueness cannot be assumed.

In [61]:
weather_df = pd.read_csv("data/raw/weather/weather.csv")
weather_key = ["year", "month", "day", "hour"]

duplicates = weather_df.duplicated(subset=weather_key).sum()

print("Duplicate records:", duplicates)
print("Nulls:")
print(weather_df[weather_key].isna().sum())

Duplicate records: 0
Nulls:
year     0
month    0
day      0
hour     0
dtype: int64


- Each combination identifies exactly one hourly weather observation.

In [62]:
air_df = pd.read_csv("data/raw/air_quality/hourly_88101_2024.csv")
air_key = [
    "State Code",
    "County Code",
    "Site Num",
    "Parameter Code",
    "POC",
    "Date Local",
    "Time Local"
]

duplicates = air_df.duplicated(subset=air_key).sum()

print("Duplicate records:", duplicates)
print("Nulls:")
print(air_df[air_key].isna().sum())

C:\Users\user\AppData\Local\Temp\ipykernel_15012\463234392.py:1: DtypeWarning: Columns (0: Qualifier) have mixed types. Specify dtype option on import or set low_memory=False.
  air_df = pd.read_csv("data/raw/air_quality/hourly_88101_2024.csv")


Duplicate records: 0
Nulls:
State Code        0
County Code       0
Site Num          0
Parameter Code    0
POC               0
Date Local        0
Time Local        0
dtype: int64


- The elements in the list `air key` can be treated as Primary Keys.

In [63]:
zones_df = pd.read_csv("data/raw/taxi_zones/taxi_zone_lookup.csv")
print("Duplicate LocationIDs:",zones_df["LocationID"].duplicated().sum())
print("Null LocationIDs:",zones_df["LocationID"].isna().sum())

Duplicate LocationIDs: 0
Null LocationIDs: 0


### Checking for Categorical Column

In [73]:
for col in ["VendorID", "RatecodeID", "store_and_fwd_flag",
            "PULocationID", "DOLocationID", "payment_type"]:
    print(f"\n{col}")
    trips.select(col).distinct().show(20)


VendorID
+--------+
|VendorID|
+--------+
|       1|
|       2|
|       6|
+--------+


RatecodeID
+----------+
|RatecodeID|
+----------+
|         2|
|        99|
|         3|
|         5|
|         4|
|         1|
|         6|
|      NULL|
+----------+


store_and_fwd_flag
+------------------+
|store_and_fwd_flag|
+------------------+
|                 N|
|                 Y|
|              NULL|
+------------------+


PULocationID
+------------+
|PULocationID|
+------------+
|         186|
|         148|
|         161|
|         107|
|         263|
|         233|
|         232|
|          13|
|         261|
|          70|
|         265|
|          14|
|          12|
|         190|
|          93|
|          18|
|         198|
|         202|
|         225|
|         157|
+------------+
only showing top 20 rows


DOLocationID
+------------+
|DOLocationID|
+------------+
|         148|
|         261|
|         190|
|         263|
|         265|
|         171|
|         107|
|         2

In [74]:
for col in [
    "temp_source",
    "rhum_source",
    "prcp_source",
    "snwd_source",
    "wdir_source",
    "wspd_source",
    "wpgt_source",
    "pres_source",
    "cldc_source",
    "coco"
]:
    print(f"\n{col}")
    print(weather_df[col].value_counts(dropna=False))


temp_source
temp_source
isd_lite    8774
metar         10
Name: count, dtype: int64

rhum_source
rhum_source
isd_lite    8774
metar         10
Name: count, dtype: int64

prcp_source
prcp_source
isd_lite      8657
dwd_mosmix     127
Name: count, dtype: int64

snwd_source
snwd_source
NaN    8784
Name: count, dtype: int64

wdir_source
wdir_source
isd_lite      8626
metar          117
dwd_mosmix      41
Name: count, dtype: int64

wspd_source
wspd_source
isd_lite      8772
metar           11
dwd_mosmix       1
Name: count, dtype: int64

wpgt_source
wpgt_source
NaN    8784
Name: count, dtype: int64

pres_source
pres_source
isd_lite      8689
metar           89
dwd_mosmix       6
Name: count, dtype: int64

cldc_source
cldc_source
isd_lite      4938
dwd_mosmix    3846
Name: count, dtype: int64

coco
coco
2.0     2916
3.0     2535
4.0     1742
7.0      578
1.0      309
8.0      291
5.0      160
9.0      152
14.0      59
25.0      10
15.0       8
12.0       7
NaN        6
16.0       5
13.0     

In [75]:
for col in [
    "Parameter Name",
    "Units of Measure",
    "Qualifier",
    "Method Type",
    "Method Code",
    "Method Name",
    "State Name",
    "County Name",
    "Datum"
]:
    print(f"\n--- {col} ---")
    print(air_df[col].value_counts(dropna=False).head(20))


--- Parameter Name ---
Parameter Name
PM2.5 - Local Conditions    8139551
Name: count, dtype: int64

--- Units of Measure ---
Units of Measure
Micrograms/cubic meter (LC)    8139551
Name: count, dtype: int64

--- Qualifier ---
Qualifier
NaN    7487648
2       147307
IT       80675
QX       72643
IF       54303
1        49889
6        44602
SX       43913
6.0      39055
IM       22568
1V       17893
J        16743
IE        8453
3         6490
IJ        5273
IH        5235
MD        4586
RF        4223
IA        3997
2.0       3871
Name: count, dtype: int64

--- Method Type ---
Method Type
FEM    8139551
Name: count, dtype: int64

--- Method Code ---
Method Code
636    2488972
170    1746149
638    1637575
209    1469204
184     228745
182      98563
581      91543
183      88558
236      69299
736      67915
195      65462
238      44547
738      34381
181       8638
Name: count, dtype: int64

--- Method Name ---
Method Name
Teledyne T640 at 5.0 LPM w/Network Data Alignment enabled - 

In [76]:
for col in [
    "State Code",
    "County Code",
    "Site Num",
    "Parameter Code",
    "POC"
]:
    print(f"\n--- {col} ---")
    print("Unique:", air_df[col].nunique(dropna=False))
    print(air_df[col].value_counts(dropna=False).head(10))


--- State Code ---
Unique: 53
State Code
6     1016518
42     465466
48     441848
39     324363
12     260456
18     238984
26     234639
49     232915
17     223128
27     203095
Name: count, dtype: int64

--- County Code ---
Unique: 126
County Code
3     408918
1     384346
13    283821
35    249795
31    225432
21    185023
19    184756
73    178931
29    160383
37    158802
Name: count, dtype: int64

--- Site Num ---
Unique: 266
Site Num
2     424475
4     348179
7     331480
3     321591
5     320757
1     272749
8     223389
10    194079
6     174100
9     171132
Name: count, dtype: int64

--- Parameter Code ---
Unique: 1
Parameter Code
88101    8139551
Name: count, dtype: int64

--- POC ---
Unique: 12
POC
3     5022900
1     1209893
4      639911
2      396352
7      292015
5      261838
6      111069
9       69399
23      61279
21      34581
Name: count, dtype: int64


## Task 2

In [53]:
from pathlib import Path

for sub in ["taxi_trips", "weather", "air_quality", "taxi_zones"]:
    files = sorted(Path("data/raw", sub).glob("*"))
    total = sum(f.stat().st_size for f in files)
    print(f"{sub:15s} {total/1e6:8.1f} MB  ({len(files)} file(s))")
    for f in files:
        print(f"    {f.name:45s} {f.stat().st_size/1e6:8.2f} MB")

taxi_trips         160.4 MB  (4 file(s))
    .gitkeep                                          0.00 MB
    yellow_tripdata_2024-01.parquet                  49.96 MB
    yellow_tripdata_2024-02.parquet                  50.35 MB
    yellow_tripdata_2024-03.parquet                  60.08 MB
weather              1.1 MB  (2 file(s))
    .gitkeep                                          0.00 MB
    weather.csv                                       1.07 MB
air_quality       2365.7 MB  (2 file(s))
    .gitkeep                                          0.00 MB
    hourly_88101_2024.csv                          2365.68 MB
taxi_zones           0.0 MB  (2 file(s))
    .gitkeep                                          0.00 MB
    taxi_zone_lookup.csv                              0.01 MB
